In [ ]:
# Choose a gemini model
from google import genai

client = genai.Client(
    api_key=""  # 只传这一个参数
)

# Load the structurized manual
from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Haystack/manual.json"
import json
with open(file_path, "r", encoding="utf-8") as file:
  manual=json.load(file)

# save the contexts, titles and pages for later use
contexts = [document["context"] for document in manual]
titles = [document["title"] for document in manual]
pages = [document["page"] for document in manual]

with open("/content/drive/MyDrive/Haystack/LDS_current_scores.json", "r", encoding="utf-8") as file:
  LDS_current_scores=json.load(file)
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_scores.json", "r", encoding="utf-8") as file:
  LDS_rewritten_scores=json.load(file)
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_queries.json", "r", encoding="utf-8") as file:
  LDS_rewritten_queries=json.load(file)
with open("/content/drive/MyDrive/Haystack/manual_QAs.json", "r", encoding="utf-8") as qas_file:
  manual_QAs = json.load(qas_file)

LDS_current_queries = [conversation["Queries"] for conversation in manual_QAs["LDS"]]

!pip install haystack-ai

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.7/643.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 4.7 MB/s eta 0:00:00


In [2]:
# Prepare the document_store, doc_embedder and text_embedder using haystack
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack import Document
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import HuggingFaceLocalGenerator
from haystack.utils import ComponentDevice

document_store = InMemoryDocumentStore()
docs = [Document(content = contexts[i], meta = {'title': titles[i], 'context_id': i, 'page': pages[i]}) for i in range(len(manual))]

doc_embedder = SentenceTransformersDocumentEmbedder(
    model="all-MiniLM-L6-v2",
    progress_bar=False
)
doc_embedder.warm_up()

docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])

text_embedder = SentenceTransformersTextEmbedder(
    model="all-MiniLM-L6-v2",
    progress_bar=False
)
text_embedder.warm_up()

retriever = InMemoryEmbeddingRetriever(document_store)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
import torch
import torch.nn.functional as F

def similarity(text1, text2):
  text1_embedding = text_embedder.run(text1)
  text2_embedding = text_embedder.run(text2)
  v1 = torch.tensor(text1_embedding["embedding"])
  v2 = torch.tensor(text2_embedding["embedding"])
  sim = F.cosine_similarity(v1, v2, dim=0)
  return sim.item()

In [43]:
LDS_rewritten_queries_and_scores = []
for i in range(40):
  LDS_rewritten_queries_and_scores_one_conversation = []
  for j in range(len(LDS_rewritten_queries[i])):
    LDS_rewritten_queries_and_scores_one_conversation.append([LDS_rewritten_queries[i][j], sum(LDS_rewritten_scores[i][j])/20])
  LDS_rewritten_queries_and_scores.append(LDS_rewritten_queries_and_scores_one_conversation)

In [122]:
i,j = 35,11
LDS_rewritten_queries_and_scores[i][j]

['Besides engaging and disengaging vertical modes, what other scenarios are described for the Autoflight System?',
 0.6700823843479157]

In [123]:
extract_keyword_rewritten(LDS_rewritten_queries_and_scores[i][j][0])

'Autoflight System scenarios, vertical modes, engage, disengag'

In [126]:
retrieval_input_embedding = text_embedder.run('Autoflight System scenarios, vertical modes, engage, disengag')
docs = retriever.run(query_embedding = retrieval_input_embedding['embedding'], top_k = 20)
keywords_ids = [docs["documents"][i].meta['context_id'] for i in range(20)]

In [127]:
all_correct_ids = [manual_QAs["LDS"][i]['Context_ids'] for i in range(40)]
keywords_score = []
for k in range(20):
  correct_id = all_correct_ids[i][j]
  correct_context = contexts[correct_id]
  keywords_context = contexts[keywords_ids[k]]
  keywords_score.append(similarity(keywords_context, correct_context))

In [128]:
sum(keywords_score)/20

0.6787835031747818

In [124]:
LDS_current_queries[i][j]

'What are some other scenarios mentioned in relation to it?'

In [125]:
sum(LDS_current_scores[i][j])/20

0.2782857157289982

In [83]:
LDS_current_queries[i][:j]

['What pages are advised for the PF and PM sides to select during the descent phase?',
 'When does this phase begin and end?',
 'What does Vertical Navigation provide in it?',
 'How does the VPATH mode achieve descent?',
 'When an altitude limitation or preselect altitude is reached during this, what happens to the altitude?',
 'How is this phase entered?',
 'Can hold procedures be set up manually during the descent phase?',
 'How is that function accessed, and how many of these can a flight plan have?',
 'What are the types of this function available in FMS?',
 'How can one of these be defined?',
 'Can a HOLD be set up at a waypoint not in the flight plan?',
 'When one is set up at such a point, what does the FMS automatically load into the flight plan?',
 'What is the initial action to configure this?',
 'After that, which page should be entered if the flight plan has one hold?',
 'What appears if there are none of them in the current flight plan after that first step?',
 'Once the L

In [103]:
def extract_keyword_rewritten(rewritten_query):
  template_for_keyword_extraction = f"""
You are an intelligent assistant for extracting keywords from a question.
Keywords will be used to retrieve relevant documents to help answering the question.
Question: {rewritten_query}
Extract 1-3 keywords (noun, verb, phrase) that you think can help retrieve relevant documents.
Answer in the format below:
Keywords: ...
"""
  model_keyword = client.models.generate_content(
      model="gemini-2.5-flash-lite",
      contents=template_for_keyword_extraction
  )
  keyword = model_keyword.text.strip("Keyword:")
  keyword = keyword.strip("Keywords:")
  keyword = keyword.strip()
  return keyword

In [131]:
import time
LDS_rewritten_keywords = []
LDS_rewritten_keywords_ids = []

for i in range(40):
  print(i, end=': ')
  rewritten_keywords = []
  rewritten_keywords_ids = []

  for j in range(len(LDS_rewritten_queries_and_scores[i])):
    print(j, end = ' ')
    keywords = extract_keyword_rewritten(LDS_rewritten_queries_and_scores[i][j][0])
    retrieval_input_embedding = text_embedder.run(keywords)
    docs = retriever.run(query_embedding = retrieval_input_embedding['embedding'], top_k = 20)
    keywords_ids = [docs["documents"][i].meta['context_id'] for i in range(20)]
    rewritten_keywords.append(keywords)
    rewritten_keywords_ids.append(keywords_ids)

  LDS_rewritten_keywords.append(rewritten_keywords)
  LDS_rewritten_keywords_ids.append(rewritten_keywords_ids)
  print()
  time.sleep(2)

with open("/content/drive/MyDrive/Haystack/LDS_rewritten_keywords.json", 'w', encoding='utf-8') as f:
  json.dump(LDS_rewritten_keywords, f)
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_keywords_ids.json", 'w', encoding='utf-8') as f:
  json.dump(LDS_rewritten_keywords_ids, f)

0: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
1: 0 1 2 3 4 5 6 7 8 
2: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
3: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
4: 0 1 2 3 4 5 6 7 8 9 10 11 
5: 0 1 2 3 4 5 6 7 
6: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
7: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
8: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 
9: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
10: 0 1 2 3 4 5 6 7 8 9 10 11 12 
11: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
12: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 
13: 0 1 2 3 4 5 6 7 8 
14: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 
15: 0 1 2 3 4 5 6 7 8 9 10 11 12 
16: 0 1 2 3 4 5 6 7 8 9 10 
17: 0 1 2 3 4 5 6 7 8 9 10 
18: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
19: 0 1 2 3 4 5 6 7 8 
20: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 
21: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
22: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 
23: 0 1 2 3 4 5 6 7

In [152]:
len(LDS_rewritten_keywords_ids)

40

In [172]:
LDS_rewritten_keywords_scores = []

for i in range(40):
  print(i, end = ": ")
  correct_ids = all_correct_ids[i]
  keywords_ids = LDS_rewritten_keywords_ids[i]
  keywords_scores = []

  for n in range(len(correct_ids)):
    print(n, end = " ")
    keywords_score = []
    for k in range(20):
      correct_id = correct_ids[n]
      keywords_id = keywords_ids[n][k]
      correct_context = contexts[correct_id]
      keywords_context = contexts[keywords_id]
      keywords_score.append(similarity(keywords_context, correct_context))
    keywords_scores.append(keywords_score)
  LDS_rewritten_keywords_scores.append(keywords_scores)
  print()

with open("/content/drive/MyDrive/Haystack/LDS_rewritten_keywords_scores.json", 'w', encoding='utf-8') as f:
  json.dump(LDS_rewritten_keywords_scores, f)

0: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
1: 0 1 2 3 4 5 6 7 8 
2: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
3: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
4: 0 1 2 3 4 5 6 7 8 9 10 11 
5: 0 1 2 3 4 5 6 7 
6: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
7: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
8: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 
9: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
10: 0 1 2 3 4 5 6 7 8 9 10 11 12 
11: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
12: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 
13: 0 1 2 3 4 5 6 7 8 
14: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 
15: 0 1 2 3 4 5 6 7 8 9 10 11 12 
16: 0 1 2 3 4 5 6 7 8 9 10 
17: 0 1 2 3 4 5 6 7 8 9 10 
18: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
19: 0 1 2 3 4 5 6 7 8 
20: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 
21: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
22: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 
23: 0 1 2 3 4 5 6 7

In [173]:
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_keywords_scores.json", "r", encoding="utf-8") as f:
  LDS_rewritten_keywords_scores = json.load(f)

In [193]:
mean_LDS_rewritten_keywords_scores = [sum(score[:5])/5 for scores in LDS_rewritten_keywords_scores for score in scores]
mean_LDS_rewritten_scores = [sum(score[:5])/5 for scores in LDS_rewritten_scores for score in scores]

In [194]:
len(mean_LDS_rewritten_keywords_scores)

652

In [195]:
len(mean_LDS_rewritten_scores)

652

In [196]:
for i in mean_LDS_rewritten_scores:
  print(f"{i:0.2f}", end=' ')

0.74 0.79 0.75 0.74 0.65 0.69 0.74 0.77 0.78 0.67 0.63 0.61 0.53 0.73 0.73 0.68 0.70 0.71 0.68 0.65 0.64 0.68 0.71 0.69 0.66 0.66 0.76 0.70 0.80 0.68 0.69 0.75 0.75 0.72 0.63 0.66 0.62 0.64 0.75 0.75 0.76 0.76 0.59 0.65 0.75 0.67 0.66 0.57 0.75 0.78 0.81 0.76 0.81 0.47 0.67 0.74 0.81 0.83 0.68 0.77 0.65 0.48 0.77 0.69 0.76 0.81 0.59 0.67 0.54 0.74 0.77 0.78 0.36 0.73 0.65 0.74 0.74 0.77 0.84 0.82 0.78 0.88 0.88 0.83 0.79 0.91 0.71 0.63 0.59 0.59 0.75 0.76 0.77 0.77 0.70 0.85 0.84 0.85 0.79 0.77 0.68 0.67 0.80 0.78 0.77 0.79 0.84 0.84 0.75 0.78 0.78 0.77 0.80 0.82 0.67 0.74 0.74 0.77 0.75 0.73 0.67 0.67 0.67 0.63 0.61 0.48 0.79 0.65 0.78 0.70 0.64 0.65 0.54 0.61 0.63 0.56 0.49 0.76 0.81 0.76 0.69 0.70 0.66 0.72 0.75 0.53 0.75 0.28 0.32 0.75 0.30 0.44 0.73 0.43 0.72 0.70 0.69 0.74 0.76 0.81 0.80 0.84 0.84 0.81 0.84 0.81 0.65 0.57 0.79 0.81 0.76 0.76 0.80 0.81 0.74 0.81 0.67 0.74 0.75 0.71 0.69 0.75 0.75 0.69 0.64 0.67 0.49 0.50 0.55 0.72 0.65 0.78 0.69 0.45 0.77 0.70 0.85 0.85 0.68 0.65 

In [197]:
for i in mean_LDS_rewritten_keywords_scores:
  print(f"{i:0.2f}", end=' ')

0.70 0.78 0.76 0.73 0.71 0.69 0.73 0.82 0.77 0.74 0.63 0.57 0.53 0.70 0.72 0.69 0.64 0.60 0.67 0.51 0.63 0.68 0.71 0.71 0.66 0.62 0.81 0.67 0.79 0.81 0.76 0.70 0.72 0.67 0.64 0.66 0.71 0.49 0.73 0.75 0.75 0.77 0.59 0.66 0.72 0.52 0.59 0.67 0.75 0.75 0.80 0.76 0.74 0.35 0.63 0.82 0.79 0.75 0.67 0.63 0.56 0.53 0.82 0.70 0.48 0.79 0.55 0.61 0.56 0.68 0.78 0.65 0.41 0.75 0.62 0.77 0.71 0.78 0.83 0.69 0.74 0.88 0.81 0.76 0.75 0.77 0.56 0.62 0.54 0.56 0.61 0.76 0.76 0.72 0.70 0.83 0.84 0.81 0.74 0.69 0.70 0.69 0.81 0.78 0.82 0.78 0.82 0.83 0.75 0.78 0.72 0.76 0.73 0.78 0.67 0.73 0.73 0.74 0.80 0.71 0.74 0.69 0.72 0.63 0.61 0.53 0.79 0.43 0.73 0.56 0.59 0.68 0.54 0.52 0.63 0.46 0.43 0.67 0.75 0.69 0.68 0.68 0.68 0.78 0.72 0.45 0.74 0.39 0.30 0.55 0.28 0.39 0.69 0.25 0.70 0.68 0.56 0.61 0.72 0.75 0.62 0.81 0.81 0.81 0.84 0.84 0.69 0.35 0.80 0.81 0.72 0.66 0.68 0.78 0.81 0.81 0.65 0.74 0.75 0.68 0.68 0.74 0.73 0.68 0.43 0.45 0.45 0.46 0.42 0.67 0.64 0.76 0.66 0.48 0.81 0.70 0.85 0.77 0.71 0.64 

In [198]:
sum(mean_LDS_rewritten_scores)/652

0.7022319211230322

In [199]:
sum(mean_LDS_rewritten_keywords_scores)/652

0.6802442734501113